In [2]:
#01:sabse pehle hum ek dataset khud banaya taki SQL apply kar sake.

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

# Ek naya "employees" sample dataset khud type karo
employees = pd.DataFrame({
    'emp_id': [1,2,3,4,5,6,7,8],
    'name': ['Aman','Priya','Rahul','Sneha','Vikas','Neha','Karan','Isha'],
    'department': ['Sales','Sales','IT','IT','HR','HR','Sales','IT'],
    'salary': [45000, 52000, 60000, 58000, 40000, 43000, 47000, 65000],
    'join_date': ['2019-01-15','2020-03-10','2018-07-22','2021-05-01','2019-11-30','2020-08-14','2022-02-20','2017-06-05']
})
employees.to_sql('employees', conn, if_exists='replace', index=False)
print(employees)

   emp_id   name department  salary   join_date
0       1   Aman      Sales   45000  2019-01-15
1       2  Priya      Sales   52000  2020-03-10
2       3  Rahul         IT   60000  2018-07-22
3       4  Sneha         IT   58000  2021-05-01
4       5  Vikas         HR   40000  2019-11-30
5       6   Neha         HR   43000  2020-08-14
6       7  Karan      Sales   47000  2022-02-20
7       8   Isha         IT   65000  2017-06-05


In [4]:
#1.Har department ka average salary nikaalo (GROUP BY).

In [5]:
q1 = pd.read_sql("""
SELECT department, AVG(salary) AS avg_salary
FROM employees
GROUP BY department;
""",conn)
print(q1)

  department  avg_salary
0         HR     41500.0
1         IT     61000.0
2      Sales     48000.0


In [6]:
#2.Sirf IT department ke employees dikhao jinki salary 55000 se zyada hai (WHERE).

In [7]:
q2 = pd.read_sql("""
SELECT *
FROM employees
WHERE department = 'IT' AND salary > 55000;
""",conn)
print(q2)

   emp_id   name department  salary   join_date
0       3  Rahul         IT   60000  2018-07-22
1       4  Sneha         IT   58000  2021-05-01
2       8   Isha         IT   65000  2017-06-05


In [8]:
#3.Har department ke andar top earner nikaalo (ROW_NUMBER() + PARTITION BY).

In [9]:
q3 = pd.read_sql("""
WITH RankEmployees as (
    SELECT *,
            ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) as rn
    FROM employees
)
SELECT emp_id, name,department,salary,join_date
FROM RankEmployees
WHERE rn = 1;
""",conn)
print(q3)

   emp_id   name department  salary   join_date
0       6   Neha         HR   43000  2020-08-14
1       8   Isha         IT   65000  2017-06-05
2       2  Priya      Sales   52000  2020-03-10


In [10]:
#4.Har employee ka apne department ke average salary se fark nikaalo (AVG() OVER (PARTITION BY ...)).

In [18]:
q4 = pd.read_sql("""
SELECT name,department,salary,
        ROUND(AVG(salary) OVER (PARTITION BY department), 2) as dept_avg_salary,
        ROUND(salary - AVG(salary) OVER (PARTITION BY department), 2) as salary_diff
FROM employees;
""",conn)
print(q4)

    name department  salary  dept_avg_salary  salary_diff
0  Vikas         HR   40000          41500.0      -1500.0
1   Neha         HR   43000          41500.0       1500.0
2  Rahul         IT   60000          61000.0      -1000.0
3  Sneha         IT   58000          61000.0      -3000.0
4   Isha         IT   65000          61000.0       4000.0
5   Aman      Sales   45000          48000.0      -3000.0
6  Priya      Sales   52000          48000.0       4000.0
7  Karan      Sales   47000          48000.0      -1000.0


In [12]:
#5.Sabse purane (senior-most) 3 employees dikhao (ORDER BY join_date).

In [20]:
q5 = pd.read_sql("""
SELECT *
FROM employees
ORDER BY join_date ASC
LIMIT 3;
""",conn)
print(q5)

   emp_id   name department  salary   join_date
0       8   Isha         IT   65000  2017-06-05
1       3  Rahul         IT   60000  2018-07-22
2       1   Aman      Sales   45000  2019-01-15


In [14]:
#6.CTE bana kar: pehle department-wise average salary nikaalo, phir un departments ko dikhao jinka average 50000 se zyada hai.

In [21]:
q6 = pd.read_sql("""
WITH DeptAvg as(
    SELECT department, AVG(salary) as avg_salary
    FROM employees
    GROUP BY department
)
SELECT department, ROUND(avg_salary,2) as avg_salary
FROM DeptAvg
WHERE avg_salary > 50000;
""",conn)
print(q6)

  department  avg_salary
0         IT     61000.0


In [16]:
#7.salary_band NTILE(3) se banao (High/Mid/Low earners groups).

In [22]:
q7 = pd.read_sql("""
WITH NtileBuckets AS (
    SELECT name, department, salary,
           NTILE(3) OVER (ORDER BY salary DESC) AS bucket
    FROM employees
)
SELECT name, department, salary,
       CASE 
           WHEN bucket = 1 THEN 'High'
           WHEN bucket = 2 THEN 'Mid'
           ELSE 'Low'
       END AS salary_band
FROM NtileBuckets;
""", conn)

print(q7)

# Kaam khatam hone par connection close karna na bhoolein
conn.close()

    name department  salary salary_band
0   Isha         IT   65000        High
1  Rahul         IT   60000        High
2  Sneha         IT   58000        High
3  Priya      Sales   52000         Mid
4  Karan      Sales   47000         Mid
5   Aman      Sales   45000         Mid
6   Neha         HR   43000         Low
7  Vikas         HR   40000         Low
